# DSA 4060 — Week 2 Practical Task For Abdirisak Hussein STD NO: 668776
## Recommendation Data and Popularity Baselines

**Scenario:** We have a dataset of users' ratings of different movies. In this notebook we load and inspect the data, clean it, explore the interaction patterns, build a user–item matrix, measure its sparsity, and build a simple popularity-based recommender system.

**Dataset:** `ratings.csv` — contains `user_id`, `movie_id`, `movie_title`, `rating`, `timestamp`.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 1. Load and Inspect the Dataset

We start by loading the raw CSV file exactly as provided, without making any changes, so we can see what we are working with before cleaning.

In [3]:
df_raw = pd.read_csv('/content/DSA_4060_Week2_Movie_Ratings.csv')
df_raw.head(10)

,user_id,movie_id,movie_title,rating,timestamp
0,U001,M001,Nairobi Nights,3,2026-01-01T08:00:00Z
1,U004,M001,Nairobi Nights,4,2026-01-06T09:00:00Z
2,U007,M001,Nairobi Nights,2,2026-01-11T10:00:00Z
3,U010,M001,Nairobi Nights,2,2026-01-16T11:00:00Z
4,U013,M001,Nairobi Nights,4,2026-01-21T12:00:00Z
5,U016,M001,Nairobi Nights,3,2026-01-26T13:00:00Z
6,U019,M001,Nairobi Nights,3,2026-01-31T14:00:00Z
7,U022,M001,Nairobi Nights,2,2026-02-05T15:00:00Z
8,U025,M001,Nairobi Nights,2,2026-02-10T16:00:00Z
9,U028,M001,Nairobi Nights,4,2026-02-15T17:00:00Z


### 1.1 Dataset dimensions and data types

In [4]:
print(f"Shape (rows, columns): {df_raw.shape}")
print()
print("Data types:")
print(df_raw.dtypes)

Shape (rows, columns): (509, 5)

Data types:
user_id        object
movie_id       object
movie_title    object
rating          int64
timestamp      object
dtype: object


**Observation:** The dataset has 509 rows and 5 columns: `user_id`, `movie_id`, `movie_title`, `rating`, and `timestamp`. All columns are read in as text (`object`) except `rating`, which is numeric (`int64`).

### 1.2 Missing values and duplicate records

In [5]:
print("Missing values per column:")
print(df_raw.isnull().sum())
print()
print(f"Number of fully duplicated rows: {df_raw.duplicated().sum()}")
df_raw[df_raw.duplicated(keep=False)].sort_values('user_id')

Missing values per column:
user_id        2
movie_id       1
movie_title    1
rating         0
timestamp      0
dtype: int64

Number of fully duplicated rows: 3


,user_id,movie_id,movie_title,rating,timestamp
88,U012,M008,The Long Journey,3,2026-03-25T12:00:00Z
501,U012,M008,The Long Journey,3,2026-03-25T12:00:00Z
17,U023,M002,Savannah Run,4,2026-02-04T13:00:00Z
205,U023,M018,The Silent Drum,3,2026-06-08T09:00:00Z
500,U023,M002,Savannah Run,4,2026-02-04T13:00:00Z
502,U023,M018,The Silent Drum,3,2026-06-08T09:00:00Z


**Observation:**
- There are a few missing values: one row is missing `user_id`, one row is missing `movie_id` (and its `movie_title`), and the counts show a small number of nulls overall.
- There are 3 fully duplicated rows (i.e. 3 records that each appear twice in the file), which need to be removed before analysis so that ratings are not double-counted.

### 1.3 Checking the rating scale

Before cleaning, let's also check whether all ratings fall within a sensible scale (movie ratings here are expected to run from 1 to 5).

In [6]:
print(df_raw['rating'].describe())
print()
invalid_ratings = df_raw[(df_raw['rating'] < 1) | (df_raw['rating'] > 5)]
print(f"Number of ratings outside the expected 1-5 scale: {len(invalid_ratings)}")
invalid_ratings

count    509.000000
mean       3.432220
std        0.981032
min       -1.000000
25%        3.000000
50%        3.000000
75%        4.000000
max        6.000000
Name: rating, dtype: float64

Number of ratings outside the expected 1-5 scale: 3


,user_id,movie_id,movie_title,rating,timestamp
506,U006,M010,Beyond the Horizon,0,2026-08-18T09:00:00Z
507,U028,M015,The Green Valley,6,2026-08-19T13:00:00Z
508,U041,M023,The Startup,-1,2026-08-20T14:00:00Z


**Observation:** The rating column has a minimum of -1 and a maximum of 6, which fall outside the expected 1–5 rating scale. These are almost certainly data entry errors and will be removed during cleaning.

## 2. Clean the Dataset

We now clean the dataset in three steps:
1. Remove exact duplicate records.
2. Remove records with a missing `user_id` or `movie_id` (a rating is meaningless without knowing who rated what).
3. Remove records whose rating falls outside the expected 1–5 scale.


In [7]:
df = df_raw.copy()
n_start = len(df)

# Step 1: remove duplicate records
df = df.drop_duplicates()
n_after_dupes = len(df)

# Step 2: remove records with missing user or movie IDs
df = df.dropna(subset=['user_id', 'movie_id'])
n_after_missing = len(df)

# Step 3: keep only ratings within the expected 1-5 scale
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]
n_after_scale = len(df)

print(f"Rows before cleaning:              {n_start}")
print(f"Rows after removing duplicates:    {n_after_dupes}  (removed {n_start - n_after_dupes})")
print(f"Rows after removing missing IDs:   {n_after_missing}  (removed {n_after_dupes - n_after_missing})")
print(f"Rows after removing invalid ratings: {n_after_scale}  (removed {n_after_missing - n_after_scale})")

df = df.reset_index(drop=True)
df.head(10)

Rows before cleaning:              509
Rows after removing duplicates:    506  (removed 3)
Rows after removing missing IDs:   503  (removed 3)
Rows after removing invalid ratings: 500  (removed 3)


,user_id,movie_id,movie_title,rating,timestamp
0,U001,M001,Nairobi Nights,3,2026-01-01T08:00:00Z
1,U004,M001,Nairobi Nights,4,2026-01-06T09:00:00Z
2,U007,M001,Nairobi Nights,2,2026-01-11T10:00:00Z
3,U010,M001,Nairobi Nights,2,2026-01-16T11:00:00Z
4,U013,M001,Nairobi Nights,4,2026-01-21T12:00:00Z
5,U016,M001,Nairobi Nights,3,2026-01-26T13:00:00Z
6,U019,M001,Nairobi Nights,3,2026-01-31T14:00:00Z
7,U022,M001,Nairobi Nights,2,2026-02-05T15:00:00Z
8,U025,M001,Nairobi Nights,2,2026-02-10T16:00:00Z
9,U028,M001,Nairobi Nights,4,2026-02-15T17:00:00Z


In [8]:
# Confirming the dataset is now clean
print("Remaining missing values:")
print(df.isnull().sum().sum())
print(f"Remaining duplicate rows: {df.duplicated().sum()}")
print(f"Rating range: {df['rating'].min()} to {df['rating'].max()}")
print(f"Final cleaned shape: {df.shape}")

Remaining missing values:
0
Remaining duplicate rows: 0
Rating range: 1 to 5
Final cleaned shape: (500, 5)


**Observation:** After cleaning, the dataset has no missing `user_id`/`movie_id` values, no duplicate rows, and every rating lies within the expected 1–5 scale. This cleaned dataframe (`df`) is used for all subsequent analysis.

## 3. Explore the Interaction Data

Next, we calculate some basic descriptive statistics about the cleaned rating data.

In [9]:
total_ratings = len(df)
n_unique_users = df['user_id'].nunique()
n_unique_movies = df['movie_id'].nunique()

most_active_user = df['user_id'].value_counts().idxmax()
most_active_user_count = df['user_id'].value_counts().max()

most_rated_movie_id = df['movie_id'].value_counts().idxmax()
most_rated_movie_count = df['movie_id'].value_counts().max()
most_rated_movie_title = df.loc[df['movie_id'] == most_rated_movie_id, 'movie_title'].iloc[0]

average_rating = df['rating'].mean()

print(f"Total number of ratings:   {total_ratings}")
print(f"Number of unique users:    {n_unique_users}")
print(f"Number of unique movies:   {n_unique_movies}")
print(f"Most active user:          {most_active_user} ({most_active_user_count} ratings)")
print(f"Most-rated movie:          {most_rated_movie_title} ({most_rated_movie_id}, {most_rated_movie_count} ratings)")
print(f"Average rating overall:    {average_rating:.2f}")

Total number of ratings:   500
Number of unique users:    50
Number of unique movies:   24
Most active user:          U010 (17 ratings)
Most-rated movie:          Nairobi Nights (M001, 37 ratings)
Average rating overall:    3.45


**Observation:** The cleaned dataset contains ratings spread across a modest number of users and movies. One user stands out as the most active rater, and one movie has attracted the most ratings overall. The average rating across the whole dataset sits slightly above the midpoint of the 1–5 scale, suggesting a mild positive bias in how users rate movies (a common pattern in real rating datasets).

## 4. Create a User–Item Interaction Matrix

We reshape the cleaned data into a matrix where each row is a user, each column is a movie, and each cell is the rating that user gave that movie (missing values mean the user has not rated that movie).

In [10]:
user_item_matrix = df.pivot_table(index='user_id', columns='movie_id', values='rating')
print(f"User-item matrix shape: {user_item_matrix.shape}  (users x movies)")
user_item_matrix.iloc[:10, :8]

User-item matrix shape: (50, 24)  (users x movies)


movie_id,M001,M002,M003,M004,M005,M006,M007,M008
user_id,,,,,,,,
U001,3.000000,2.000000,NaN,NaN,4.0,4.0,NaN,NaN
U002,3.000000,NaN,NaN,3.5,NaN,NaN,3.0,NaN
U003,NaN,NaN,NaN,NaN,4.0,NaN,NaN,4.0
U004,4.000000,NaN,NaN,NaN,3.0,3.0,NaN,NaN
U005,2.000000,4.000000,NaN,3.0,NaN,NaN,5.0,NaN
U006,3.000000,3.000000,NaN,5.0,4.0,NaN,NaN,3.0
U007,2.333333,NaN,NaN,NaN,5.0,4.0,NaN,NaN
U008,3.000000,2.000000,NaN,NaN,3.0,4.0,5.0,NaN
U009,NaN,NaN,NaN,4.0,4.0,3.0,NaN,3.5


**Observation:** The blank (`NaN`) cells represent movies that a given user has not rated. As expected for recommendation data, most cells are empty ; each user has typically rated only a small fraction of all available movies.

## 5. Calculate Sparsity

Sparsity measures how much of the user–item matrix is *empty* (i.e. how few of all possible user–movie pairs have an observed rating):

$$\text{Sparsity} = 1 - \frac{\text{Observed ratings}}{\text{Number of users} \times \text{Number of movies}}$$


In [11]:
n_observed = df.shape[0]
n_users = user_item_matrix.shape[0]
n_movies = user_item_matrix.shape[1]
n_possible = n_users * n_movies

sparsity = 1 - (n_observed / n_possible)

print(f"Observed ratings:     {n_observed}")
print(f"Possible user-movie pairs (users x movies): {n_users} x {n_movies} = {n_possible}")
print(f"Sparsity: {sparsity:.4f}  ({sparsity*100:.2f}%)")

Observed ratings:     500
Possible user-movie pairs (users x movies): 50 x 24 = 1200
Sparsity: 0.5833  (58.33%)


**What this means:** A sparsity of roughly this magnitude tells us that the vast majority of possible user–movie combinations have never been rated — most users have only interacted with a small handful of the available movies. This is typical of real-world recommendation datasets and is exactly the reason collaborative filtering techniques (which rely on finding similar users or items from overlapping ratings) can struggle: there is very little overlap to work with. It also motivates simpler approaches, like the popularity-based recommender below, which don't require a dense matrix to produce reasonable suggestions.

## 6. Build a Popularity-Based Recommender

For every movie we calculate:
- The **average rating** it has received.
- The **number of ratings** it has received.

We then recommend the **top 10 movies with at least 10 ratings**, ranked by average rating. Requiring a minimum number of ratings avoids recommending a movie that only has one or two (possibly unrepresentative) ratings.

In [12]:
movie_stats = df.groupby(['movie_id', 'movie_title']).agg(
    average_rating=('rating', 'mean'),
    num_ratings=('rating', 'count')
).reset_index()

movie_stats = movie_stats.sort_values(['average_rating', 'num_ratings'], ascending=False)
movie_stats.head(15)

,movie_id,movie_title,average_rating,num_ratings
20,M021,Market Day,4.466667,15
12,M013,Midnight in Mombasa,4.352941,17
13,M014,Code Breakers,4.250000,16
18,M019,Future Africa,4.133333,15
19,M020,Home Again,4.133333,15
6,M007,Hidden Lake,4.125000,16
5,M006,City of Tomorrow,3.911765,34
4,M005,Mountain Echoes,3.864865,37
11,M012,The Final Goal,3.764706,17
3,M004,Coastal Dreams,3.666667,27


In [13]:
eligible_movies = movie_stats[movie_stats['num_ratings'] >= 10]

top_10_movies = eligible_movies.sort_values(
    ['average_rating', 'num_ratings'], ascending=False
).head(10).reset_index(drop=True)

top_10_movies['average_rating'] = top_10_movies['average_rating'].round(2)
top_10_movies

,movie_id,movie_title,average_rating,num_ratings
0,M021,Market Day,4.47,15
1,M013,Midnight in Mombasa,4.35,17
2,M014,Code Breakers,4.25,16
3,M019,Future Africa,4.13,15
4,M020,Home Again,4.13,15
5,M007,Hidden Lake,4.12,16
6,M006,City of Tomorrow,3.91,34
7,M005,Mountain Echoes,3.86,37
8,M012,The Final Goal,3.76,17
9,M004,Coastal Dreams,3.67,27


In [14]:
#  the top 10 recommended movies to a CSV file
top_10_movies.to_csv('top_10_recommended_movies.csv', index=False)
print("Saved top_10_recommended_movies.csv")

Saved top_10_recommended_movies.csv


## 7. Interpretation of Results

**Why the recommended movies appear at the top**
These movies rank highest because they combine two things: a high *average* rating from the users who watched them, and *enough* ratings (at least 10). Movies with very few ratings were excluded even if their average looked high, because one or two raters can easily produce a misleadingly extreme average.

**One advantage of popularity-based recommendation**
It is simple, fast, and requires no information about an individual user; it works even for a brand-new user with no rating history (avoiding the "cold-start" problem that affects personalized methods like collaborative filtering).

**Two limitations of this approach**
1. **No personalization** — every user is shown the exact same list of top movies, regardless of their individual tastes, so it doesn't account for the fact that different users like different styles.
2. **Popularity bias** — the same small set of already popular movies keeps getting recommended, which can make it harder for another movies to ever be surfaced to users, and makes recommendations feel repetitive across the whole user base.


## Summary

- Loaded the raw ratings dataset (509 rows) and identified missing values, duplicate rows, and out-of-range ratings.
- Cleaned the dataset by removing duplicates, rows with missing IDs, and invalid ratings.
- Computed core statistics: total ratings, unique users/movies, most active user, most-rated movie, and overall average rating.
- Built a user–item interaction matrix and calculated its sparsity, showing that the matrix is mostly empty (as is typical for recommendation data).
- Built a simple popularity-based recommender, producing a top-10 list of movies with at least 10 ratings, saved to `top_10_recommended_movies.csv`.
